# Chemistry-catalog corollary: $\widehat B^{\mathrm{charge}}_4(W) \in \{1, 3, 5\}$

Manuscript reference: Section III, Corollary 1.

For each of the five catalog word types, we directly enumerate the partitions in $\Pi^{\mathrm{nl}}_{|W|}(W)$ (charge-neutral with at least one block of size $> 2$), find the per-partition minimizer of the distinguished large block, and sum the resulting $\prod_{B \ne B_\star} M_{|B|}$ products. The manuscript values are

| word type | $\widehat B^{\mathrm{charge}}_4$ |
|---|---:|
| $n_i n_j n_k$ | $1$ |
| $a^\dagger_i a_j n_k$ | $1$ |
| $a^\dagger_i a_j n_k n_\ell$ | $3$ |
| $a^\dagger_i a^\dagger_j a_k a_\ell$ | $1$ |
| $n_i n_j n_k n_\ell$ | $5$ |

In [1]:
from connected_layer_sector import M_r_const, set_partitions
from connected_layer_sector.operators import letter_charge


def block_charge(word, block):
    return sum(letter_charge(word[j]) for j in block)


def neutral_large_partitions(word):
    """Yield every partition pi of [|word|] whose blocks are all
    charge-neutral and at least one block has size > 2."""
    m = len(word)
    for pi in set_partitions(range(m)):
        if not any(len(b) > 2 for b in pi):
            continue
        if any(block_charge(word, b) != 0 for b in pi):
            continue
        yield pi


def block_refined_constant(word, r=4):
    """Direct enumeration of
        sum_{pi in Pi^nl_m(W)} min_{B_star: |B_star|>2} prod_{B != B_star} M_{|B|}.
    """
    total = 0
    for pi in neutral_large_partitions(word):
        # candidates = blocks B_star with |B_star| > 2
        candidates = [B for B in pi if len(B) > 2]
        per_choice = []
        for B_star in candidates:
            prod = 1
            for B in pi:
                if B is B_star:
                    continue
                prod *= M_r_const(len(B))
            per_choice.append(prod)
        total += min(per_choice)
    return total

## Evaluate on each catalog word type

Use distinct site indices (the catalog requires distinct site indices — Definition 3 item 2).

In [2]:
catalog_examples = [
    ("n_i n_j n_k", (("n", 0), ("n", 1), ("n", 2))),
    ("a^d_i a_j n_k", (("ad", 0), ("a", 1), ("n", 2))),
    ("a^d_i a_j n_k n_l", (("ad", 0), ("a", 1), ("n", 2), ("n", 3))),
    ("a^d_i a^d_j a_k a_l", (("ad", 0), ("ad", 1), ("a", 2), ("a", 3))),
    ("n_i n_j n_k n_l", (("n", 0), ("n", 1), ("n", 2), ("n", 3))),
]

manuscript = {
    "n_i n_j n_k": 1,
    "a^d_i a_j n_k": 1,
    "a^d_i a_j n_k n_l": 3,
    "a^d_i a^d_j a_k a_l": 1,
    "n_i n_j n_k n_l": 5,
}

print(f"{'word type':<24} {'B_hat^charge_4':>15}  manuscript")
print("-" * 55)
for label, word in catalog_examples:
    val = block_refined_constant(word, r=4)
    expected = manuscript[label]
    flag = "OK" if val == expected else "MISMATCH"
    print(f"{label:<24} {val:>15}  {expected}  {flag}")

word type                 B_hat^charge_4  manuscript
-------------------------------------------------------
n_i n_j n_k                            1  1  OK
a^d_i a_j n_k                          1  1  OK
a^d_i a_j n_k n_l                    3.0  3  OK
a^d_i a^d_j a_k a_l                    1  1  OK
n_i n_j n_k n_l                      5.0  5  OK


## Show which partitions contribute (length 4, $n_i n_j n_k n_\ell$)

Manuscript: 1 (top partition) + 4 (each $1+3$ split) = 5.

In [3]:
word = (("n", 0), ("n", 1), ("n", 2), ("n", 3))
print("Neutral-large partitions of n_i n_j n_k n_l:")
for pi in neutral_large_partitions(word):
    pi_str = "  |  ".join("{" + ", ".join(str(j) for j in sorted(b)) + "}" for b in pi)
    candidates = [B for B in pi if len(B) > 2]
    contributions = []
    for B_star in candidates:
        prod = 1
        for B in pi:
            if B is B_star:
                continue
            prod *= M_r_const(len(B))
        contributions.append(prod)
    print(f"  pi = {pi_str}    contribution = {min(contributions)}")

print()
print(f"sum = {block_refined_constant(word, r=4)} (manuscript: 5)")

Neutral-large partitions of n_i n_j n_k n_l:
  pi = {0, 1, 2, 3}    contribution = 1
  pi = {0}  |  {1, 2, 3}    contribution = 1.0
  pi = {1}  |  {0, 2, 3}    contribution = 1.0
  pi = {0, 1, 2}  |  {3}    contribution = 1.0
  pi = {2}  |  {0, 1, 3}    contribution = 1.0

sum = 5.0 (manuscript: 5)


## Show which partitions contribute (length 4, $a^\dagger_i a_j n_k n_\ell$)

Manuscript: 1 (top) + 2 ($3+1$ partitions whose singleton is one of the two zero-charge letters) = 3. The other $3+1$ partitions whose singleton is one of the charged letters $a^\dagger$ or $a$ are charge-non-neutral and excluded.

In [4]:
word = (("ad", 0), ("a", 1), ("n", 2), ("n", 3))
print("Neutral-large partitions of a^d_i a_j n_k n_l:")
for pi in neutral_large_partitions(word):
    pi_str = "  |  ".join("{" + ", ".join(str(j) for j in sorted(b)) + "}" for b in pi)
    candidates = [B for B in pi if len(B) > 2]
    contributions = [
        sum(M_r_const(len(B)) for B in pi if B is not B_star) or 1
        for B_star in candidates
    ]
    # use product, not sum:
    contributions = []
    for B_star in candidates:
        prod = 1
        for B in pi:
            if B is B_star:
                continue
            prod *= M_r_const(len(B))
        contributions.append(prod)
    print(f"  pi = {pi_str}    contribution = {min(contributions)}")

print()
print(f"sum = {block_refined_constant(word, r=4)} (manuscript: 3)")

Neutral-large partitions of a^d_i a_j n_k n_l:
  pi = {0, 1, 2, 3}    contribution = 1
  pi = {0, 1, 2}  |  {3}    contribution = 1.0
  pi = {2}  |  {0, 1, 3}    contribution = 1.0

sum = 3.0 (manuscript: 3)


## Sanity-check assertions

In [5]:
for label, word in catalog_examples:
    val = block_refined_constant(word, r=4)
    expected = manuscript[label]
    assert val == expected, f"{label}: enumerated {val}, manuscript {expected}"

# Same values at r=5: no partition of [m<=4] has a block of size 5,
# so M_|B| <= M_4 = 26 in every non-distinguished block; the values
# computed with M_r_const(5) at r=5 must equal the r=4 values.
for label, word in catalog_examples:
    assert block_refined_constant(word, r=5) == block_refined_constant(word, r=4)

print("PASS: {1, 1, 3, 1, 5} block-refined values reproduced for the chemistry catalog.")

PASS: {1, 1, 3, 1, 5} block-refined values reproduced for the chemistry catalog.
